# 3. diena - JSON, CSV un Pandas pamati

## Notebook mērķis

Šajā notebook soli pa solim apskatīsim, kā:
- nolasīt vienkāršu `JSON` failu ar Python standarta bibliotēku
- apskatīt `JSON` struktūru
- strādāt ar ligzdotu `JSON`
- nolasīt `CSV` failus ar `pandas`
- apskatīt `DataFrame`
- atlasīt, filtrēt, kārtot un pārveidot datus
- apvienot tabulas ar `concat()` un `merge()`
- eksportēt rezultātus uz `CSV` un `JSON`

## Šajā nodarbībā izmantotie faili

Mēs izmantosim paraugdatus no mapes `data/day3/`:
- `products_basic.json`
- `orders_nested.json`
- `products.csv`
- `sales_january.csv`
- `sales_february.csv`

## Svarīgi

Izpildiet notebook no augšas uz leju. Vēlākās šūnas izmanto mainīgos, kas
izveidoti iepriekšējās šūnās.


In [ ]:
from __future__ import annotations

import json
from pathlib import Path

try:
    import pandas as pd
except ModuleNotFoundError as exc:
    raise SystemExit(
        "Šim notebook ir vajadzīgs pandas. Vispirms instalējiet to, piemēram: pip install pandas"
    ) from exc


def find_repo_root() -> Path:
    """Atrod projekta saknes mapi gan skripta, gan Jupyter izpildes gadījumā."""
    candidates: list[Path] = []

    try:
        candidates.append(Path(__file__).resolve().parents[1])
    except NameError:
        pass

    cwd = Path.cwd().resolve()
    candidates.extend([cwd, cwd.parent])

    for candidate in candidates:
        if (candidate / "data" / "day3").exists():
            return candidate

    return cwd


REPO_ROOT = find_repo_root()
DATA_DIR = REPO_ROOT / "data" / "day3"


## 1. Pārbaudām, vai datu faili eksistē

Pirms datu nolasīšanas ir laba prakse pārbaudīt failu ceļus.


In [ ]:
data_files = sorted(DATA_DIR.glob("*"))
print("Datu mape:", DATA_DIR.resolve())
print("Atrasti faili:")
for path in data_files:
    print("-", path.name)


## 2. Nolasām vienkāršu JSON failu

`products_basic.json` ir vienkāršs, plakans `JSON` fails.
Tas satur produktu objektu sarakstu.
Šādu struktūru ir visvieglāk pārveidot tabulā.


In [ ]:
products_json_path = DATA_DIR / "products_basic.json"

with products_json_path.open("r", encoding="utf-8") as file:
    products_data = json.load(file)

print("Python tips:", type(products_data))
print("Ierakstu skaits:", len(products_data))


Apskatīsim vienu elementu no saraksta.


In [ ]:
products_data[0]


Katrs elements ir Python vārdnīca.


In [ ]:
print("Pirmā elementa tips:", type(products_data[0]))
print("Pirmā elementa atslēgas:", list(products_data[0].keys()))


## 3. Ejam cauri JSON ierakstiem ar ciklu

Pirms lietot `pandas`, ir vērts apskatīt neapstrādātos datus tieši Python
struktūrās.


In [ ]:
for product in products_data[:3]:
    print(product["product_id"], "-", product["name"], "-", product["price"])


## 4. Pārveidojam vienkāršu JSON par DataFrame

Sarakstu ar vārdnīcām bieži var uzreiz pārvērst par `pandas DataFrame`.


In [ ]:
products_df = pd.DataFrame(products_data)
products_df


## 5. Apskatām DataFrame

Šīs ir pirmās komandas, ar kurām vajadzētu kļūt ērtāk strādāt.


In [ ]:
products_df.head()


In [ ]:
products_df.shape


In [ ]:
products_df.columns


In [ ]:
products_df.dtypes


In [ ]:
products_df.info()


## 6. Vienkārša kolonnu atlase

Varam atlasīt vienu kolonnu vai vairākas kolonnas.


In [ ]:
products_df["name"]


In [ ]:
products_df[["product_id", "name", "category", "price"]]


## 7. Filtrējam rindas

Šeit atlasām tikai elektronikas produktus.


In [ ]:
products_df[products_df["category"] == "Electronics"]


Varam kombinēt arī vairākus nosacījumus.


In [ ]:
products_df[(products_df["category"] == "Electronics") & (products_df["price"] > 10)]


## 8. Kārtojam vērtības

Kārtošana ir viens no pirmajiem praktiskajiem datu analīzes uzdevumiem.


In [ ]:
products_df.sort_values("price")


In [ ]:
products_df.sort_values("price", ascending=False)


## 9. Izveidojam jaunas kolonnas

Jaunas vērtības varam aprēķināt no jau esošajām kolonnām.


In [ ]:
products_df["price_with_vat"] = (products_df["price"] * 1.21).round(2)
products_df["name_upper"] = products_df["name"].str.upper()
products_df


## 10. Grupējam un veidojam kopsavilkumu

Šis ir līdzīgi vienkāršai Excel PivotTable pieejai.


In [ ]:
products_df.groupby("category")["price"].mean().sort_values(ascending=False)


In [ ]:
products_summary = (
    products_df.groupby("category")
    .agg(
        product_count=("product_id", "count"),
        average_price=("price", "mean"),
        max_price=("price", "max"),
    )
    .round(2)
)
products_summary


## 11. Nolasām ligzdotu JSON failu

`orders_nested.json` ir sarežģītāks piemērs.
Šis fails nav tikai saraksts. Tas ir objekts ar metadatiem un ar `orders`
sarakstu tā iekšpusē.


In [ ]:
orders_json_path = DATA_DIR / "orders_nested.json"

with orders_json_path.open("r", encoding="utf-8") as file:
    orders_document = json.load(file)

print("Augšējā līmeņa tips:", type(orders_document))
print("Augšējā līmeņa atslēgas:", list(orders_document.keys()))


Apskatīsim atskaites metadatus.


In [ ]:
print("Atskaites nosaukums:", orders_document["report_name"])
print("Izveidots:", orders_document["generated_at"])
print("Valūta:", orders_document["currency"])


Paši ieraksti atrodas laukā `orders`.


In [ ]:
orders_list = orders_document["orders"]
print("Pasūtījumu skaits:", len(orders_list))
orders_list[0]


## 12. Piekļuve ligzdotām vērtībām manuāli

Tas ir noderīgi, kamēr vēl tikai mācāmies saprast `JSON` struktūras.


In [ ]:
first_order = orders_list[0]

print("Pasūtījuma ID:", first_order["order_id"])
print("Klienta vārds:", first_order["customer"]["name"])
print("Klienta pilsēta:", first_order["customer"]["city"])
print("Maksājuma statuss:", first_order["payment"]["status"])
print("Pirmā pasūtītā prece:", first_order["items"][0]["product_id"])


## 13. Izlīdzinām ligzdotu JSON ar ciklu

Šeit veidojam vienu rindu katram pasūtījumam.
Dažiem laukiem apzināti izmantojam `.get()`, jo ne visiem pasūtījumiem ir
pilnīgi vienāda struktūra.


In [ ]:
order_rows = []

for order in orders_list:
    items = order.get("items", [])
    total_quantity = sum(item["quantity"] for item in items)
    items_total = sum(item["quantity"] * item["unit_price"] for item in items)
    delivery_cost = order.get("delivery", {}).get("cost", 0)
    total_amount = round(items_total + delivery_cost, 2)

    order_rows.append(
        {
            "order_id": order["order_id"],
            "customer_id": order["customer"]["customer_id"],
            "customer_name": order["customer"]["name"],
            "city": order["customer"]["city"],
            "loyalty_tier": order["customer"].get("loyalty_tier"),
            "item_count": len(items),
            "total_quantity": total_quantity,
            "items_total": round(items_total, 2),
            "delivery_type": order.get("delivery", {}).get("type"),
            "delivery_cost": delivery_cost,
            "payment_method": order.get("payment", {}).get("method"),
            "payment_status": order.get("payment", {}).get("status"),
            "coupon_code": order.get("coupon_code"),
            "total_amount": total_amount,
        }
    )

orders_df = pd.DataFrame(order_rows)
orders_df


## 14. Apskatām trūkstošās vērtības

Pēc ligzdota `JSON` izlīdzināšanas bieži parādās trūkstošas vērtības.


In [ ]:
orders_df.isna().sum()


Piemēram, ne visiem pasūtījumiem ir `coupon_code` vai `loyalty_tier`.


In [ ]:
orders_df[["order_id", "loyalty_tier", "coupon_code", "delivery_cost"]]


## 15. Filtrējam rezultātu no ligzdotā JSON

Šeit atstājam tikai apmaksātos pasūtījumus, kuros ir vismaz viena prece.


In [ ]:
paid_orders_df = orders_df[
    (orders_df["payment_status"] == "paid") & (orders_df["item_count"] > 0)
]
paid_orders_df


## 16. Nolasām CSV failus ar pandas

`CSV` jau ir tabulveida formāts, tāpēc `pandas` to var nolasīt tieši.


In [ ]:
sales_january_path = DATA_DIR / "sales_january.csv"
sales_february_path = DATA_DIR / "sales_february.csv"
products_csv_path = DATA_DIR / "products.csv"

sales_january_df = pd.read_csv(sales_january_path)
sales_february_df = pd.read_csv(sales_february_path)
products_lookup_df = pd.read_csv(products_csv_path)


In [ ]:
sales_january_df.head()


In [ ]:
sales_january_df.shape


In [ ]:
sales_january_df.dtypes


In [ ]:
sales_january_df.info()


## 17. Pārveidojam kolonnu datu tipus

Datumu kolonnas parasti jāpārveido atbilstošā tipā.


In [ ]:
sales_january_df["date"] = pd.to_datetime(sales_january_df["date"])
sales_february_df["date"] = pd.to_datetime(sales_february_df["date"])

sales_january_df.dtypes


## 18. Trūkstošās vērtības CSV datos

Pamaniet, ka kolonnā `discount_pct` ir trūkstošas vērtības.


In [ ]:
sales_january_df.isna().sum()


In [ ]:
sales_january_df[sales_january_df["discount_pct"].isna()]


Šīs trūkstošās atlaides varam aizpildīt ar `0`.


In [ ]:
sales_january_df["discount_pct"] = sales_january_df["discount_pct"].fillna(0)
sales_february_df["discount_pct"] = sales_february_df["discount_pct"].fillna(0)

sales_january_df.isna().sum()


## 19. Atlasām un filtrējam CSV datus


In [ ]:
sales_january_df[["sale_id", "date", "product_id", "store", "units_sold"]]


In [ ]:
sales_january_df[sales_january_df["units_sold"] >= 10]


In [ ]:
sales_january_df[sales_january_df["store"] == "Riga-Center"]


## 20. Veidojam aprēķinātās kolonnas

Vispirms pievienojam produktu cenas pārdošanas tabulai ar `merge()`.


In [ ]:
sales_january_enriched_df = sales_january_df.merge(
    products_lookup_df[["product_id", "name", "category", "price"]],
    on="product_id",
    how="left",
)

sales_january_enriched_df.head()


Tagad varam aprēķināt bruto ieņēmumus un ieņēmumus pēc atlaides.


In [ ]:
sales_january_enriched_df["gross_revenue"] = (
    sales_january_enriched_df["units_sold"] * sales_january_enriched_df["price"]
).round(2)

sales_january_enriched_df["net_revenue"] = (
    sales_january_enriched_df["gross_revenue"]
    * (1 - sales_january_enriched_df["discount_pct"] / 100)
).round(2)

sales_january_enriched_df.head()


## 21. Kārtojam un veidojam pārdošanas kopsavilkumus


In [ ]:
sales_january_enriched_df.sort_values("net_revenue", ascending=False)


In [ ]:
sales_by_store_df = (
    sales_january_enriched_df.groupby("store")
    .agg(
        total_units=("units_sold", "sum"),
        total_revenue=("net_revenue", "sum"),
        average_discount=("discount_pct", "mean"),
    )
    .round(2)
    .sort_values("total_revenue", ascending=False)
)
sales_by_store_df


In [ ]:
sales_by_category_df = (
    sales_january_enriched_df.groupby("category")
    .agg(
        total_units=("units_sold", "sum"),
        total_revenue=("net_revenue", "sum"),
    )
    .round(2)
    .sort_values("total_revenue", ascending=False)
)
sales_by_category_df


## 22. Apvienojam mēnešu tabulas ar concat()

Janvāra un februāra failiem ir vienādas kolonnas, tāpēc vertikāla
apvienošana ir vienkārša.


In [ ]:
all_sales_df = pd.concat([sales_january_df, sales_february_df], ignore_index=True)
all_sales_df


In [ ]:
all_sales_df.shape


## 23. Apvienojam pārdošanas datus ar produktu tabulu

`merge()` apvieno divas tabulas pēc kopīgas atslēgas.


In [ ]:
all_sales_enriched_df = all_sales_df.merge(products_lookup_df, on="product_id", how="left")
all_sales_enriched_df.head()


Pievienosim aprēķinātās ieņēmumu kolonnas arī apvienotajiem datiem.


In [ ]:
all_sales_enriched_df["gross_revenue"] = (
    all_sales_enriched_df["units_sold"] * all_sales_enriched_df["price"]
).round(2)

all_sales_enriched_df["net_revenue"] = (
    all_sales_enriched_df["gross_revenue"]
    * (1 - all_sales_enriched_df["discount_pct"] / 100)
).round(2)

all_sales_enriched_df.head()


## 24. Analīze pa abiem mēnešiem kopā

Tagad varam veidot kopsavilkumus par abiem mēnešiem.


In [ ]:
monthly_store_summary_df = (
    all_sales_enriched_df.groupby("store")
    .agg(
        total_units=("units_sold", "sum"),
        total_revenue=("net_revenue", "sum"),
        row_count=("sale_id", "count"),
    )
    .round(2)
    .sort_values("total_revenue", ascending=False)
)
monthly_store_summary_df


In [ ]:
product_summary_df = (
    all_sales_enriched_df.groupby(["category", "name"])
    .agg(
        total_units=("units_sold", "sum"),
        total_revenue=("net_revenue", "sum"),
    )
    .round(2)
    .sort_values(["category", "total_revenue"], ascending=[True, False])
)
product_summary_df


## 25. Vienkārša pivot tabula

Šis ir viens no biežākajiem soļiem Excel lietotājiem, kuri sāk strādāt ar
`pandas`.


In [ ]:
pivot_table_df = pd.pivot_table(
    all_sales_enriched_df,
    index="store",
    columns="category",
    values="net_revenue",
    aggfunc="sum",
    fill_value=0,
).round(2)

pivot_table_df


## 26. JSON un CSV salīdzinājums

Īss salīdzinājums:
- `CSV` ir ērtāks, ja dati jau ir tabulas formā
- vienkāršu `JSON` bieži var tieši pārveidot par tabulu
- ligzdotu `JSON` parasti vispirms vajag apskatīt un izlīdzināt

Zemāk salīdzinām trīs galveno tabulu izmērus.


In [ ]:
print("products_df izmērs:", products_df.shape)
print("orders_df izmērs:", orders_df.shape)
print("all_sales_enriched_df izmērs:", all_sales_enriched_df.shape)


## 27. Rezultātu eksports

Reālos darba procesos notīrītos vai apkopotos datus bieži saglabājam failos.


In [ ]:
output_dir = DATA_DIR / "outputs"
output_dir.mkdir(exist_ok=True)

sales_by_store_df.to_csv(output_dir / "sales_by_store_january.csv")
monthly_store_summary_df.to_csv(output_dir / "sales_by_store_all_months.csv")
orders_df.to_json(output_dir / "orders_flattened.json", orient="records", indent=2)

print("Faili saglabāti mapē:", output_dir.resolve())


## 28. Idejas patstāvīgai praksei

Pamēģiniet paši:
- atlasīt tikai `Office` kategorijas produktus
- atrast visdārgāko produktu
- aprēķināt kopējos janvāra ieņēmumus pa produktiem
- atrast pasūtījumus, kuriem trūkst `loyalty_tier`
- sakārtot visus pārdošanas ierakstus pēc `units_sold`
- eksportēt tikai Electronics pārdošanas datus atsevišķā `CSV` failā

Zem šīs vietas varat pievienot savas šūnas.


In [ ]:
# Rakstiet savu prakses kodu šeit.
